# Collations and ANSI Mode

Two changes in Spark 4 that almost every data engineer hits on a 3 → 4 upgrade:

* **Collations** — string columns can now carry a collation (e.g. `UTF8_LCASE`, `de_DE`, ...). Equality, sorting and aggregation all respect that collation, so case-insensitive or locale-aware comparisons no longer require sprinkling `lower()` / `upper()` everywhere.
* **ANSI mode on by default** — arithmetic overflow, division by zero and invalid casts now raise instead of silently returning `null`. The `try_*` family (`try_add`, `try_multiply`, `try_divide`, `try_cast`, `try_to_number`, ...) is the explicit, opt-in way to keep lenient semantics where you actually want them.

Tasks:
1. Primer — comparing strings with and without a collation.
2. Case-insensitive aggregation on the users dataset.
3. Case-insensitive filtering without `lower()`.
4. Locale-aware ordering.
5. Verify ANSI mode is the new default.
6. Division by zero — and `try_divide`.
7. Invalid cast — and `try_cast`.
8. Arithmetic overflow — and `try_multiply` as the safe alternative.
9. Lenient numeric parsing with `try_to_number`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, collate, desc, lit, try_multiply, try_divide, try_to_number
)

import os

In [ ]:
spark = (
    SparkSession
    .builder
    .appName('Collations and ANSI Mode')
    .getOrCreate()
)

In [ ]:
print(spark.version)

In [ ]:
base_path = os.getcwd()

project_path = ('/').join(base_path.split('/')[0:-3]) 

users_input_path = os.path.join(project_path, 'data/users')

usersDF = spark.read.parquet(users_input_path)

## Part I — Collations

### Task 1: Primer — how a collation changes string equality

By default a string column uses the `UTF8_BINARY` collation, where `'Prague'` and `'prague'` compare as different. Applying `UTF8_LCASE` makes equality case-insensitive.

Hint:
* in SQL the syntax is `<string> COLLATE <name>`
* in the DataFrame API use `collate(col, '<name>')`
* docs for [collation support](https://spark.apache.org/docs/latest/sql-ref-collation.html)

In [ ]:
spark.sql("SELECT 'Prague' = 'prague' AS default_eq").show()

spark.sql("SELECT 'Prague' COLLATE UTF8_LCASE = 'prague' COLLATE UTF8_LCASE AS ci_eq").show()

### Task 2: Case-insensitive aggregation on the users dataset

Count distinct values of `location` first with the default binary collation, then with `UTF8_LCASE`. If the dataset contains any case variations of the same location, the second number will be lower.

Hint:
* `usersDF.select('location').distinct().count()` — binary
* `usersDF.select(collate('location', 'UTF8_LCASE')).distinct().count()` — case-insensitive
* the column-level collation could also be declared in the schema as `STRING COLLATE UTF8_LCASE`, in which case every operation on it would be case-insensitive automatically

In [ ]:
default_count = (
    usersDF
    .filter(col('location').isNotNull())
    .select('location')
    .distinct()
    .count()
)

ci_count = (
    usersDF
    .filter(col('location').isNotNull())
    .select(collate('location', 'UTF8_LCASE'))
    .distinct()
    .count()
)

print('distinct locations (binary):           ', default_count)
print('distinct locations (UTF8_LCASE):       ', ci_count)
print('collapsed by case-insensitive matching:', default_count - ci_count)

### Task 3: Case-insensitive filtering without `lower()`

Find all users whose location matches `'London'` regardless of case, by applying `UTF8_LCASE` to the column and comparing against a plain string literal.

Hint:
* `usersDF.filter(collate('location', 'UTF8_LCASE') == 'london')`
* the comparison takes the collation of the left-hand side
* compare against the equivalent `lower(col('location')) == 'london'` formulation — the collation version expresses intent more directly and lets the engine reason about the column's collation

In [ ]:
(
    usersDF
    .filter(collate('location', 'UTF8_LCASE') == 'london')
    .select('user_id', 'location', 'reputation')
).show(n=5, truncate=40)

### Task 4: Locale-aware ordering

Different languages have different sorting rules — for example, Czech treats `ch` as a single letter that sorts after `h`, German has multiple ways of ordering umlauts, and Swedish orders `å`, `ä`, `ö` at the very end of the alphabet. Sort the distinct locations using a Czech locale (`cs`) so the ordering reflects Czech alphabet rules instead of raw Unicode code points.

Hint:
* `orderBy(collate('location', 'cs'))`

In [ ]:
(
    usersDF
    .filter(col('location').isNotNull())
    .select('location')
    .distinct()
    .orderBy('location')
).show(n=20, truncate=40)

In [ ]:
(
    usersDF
    .filter(col('location').isNotNull())
    .select('location')
    .distinct()
    .orderBy(collate('location', 'cs'))
).show(n=20, truncate=40)

## Part II — ANSI Mode and `try_*` Functions

### Task 5: Verify ANSI mode is the new default

In Spark 4 `spark.sql.ansi.enabled` is `true` out of the box. Under ANSI semantics, the engine refuses to silently produce a wrong answer — integer overflow, division by zero, and invalid casts all raise instead of returning `null` or a wrapped value.

Hint:
* read the current value with `spark.conf.get('spark.sql.ansi.enabled')`
* it can be turned off (`SET spark.sql.ansi.enabled = false`) but that should be a deliberate, scoped decision — not the default for new pipelines

In [ ]:
print('spark.sql.ansi.enabled =', spark.conf.get('spark.sql.ansi.enabled'))

### Task 6: Division by zero

Integer division by zero now raises. `try_divide` returns `null` for the offending rows so the rest of the query still completes.

Hint:
* failing: `SELECT 10 / 0`
* safe: `try_divide(10, 0)`

In [ ]:
spark.sql('SELECT 10 / 0 AS x').show()

In [ ]:
# spark.conf.set('spark.sql.ansi.enabled', False)

In [ ]:
(
    spark
    .createDataFrame([(10, 2), (10, 0), (5, 5)], ['a', 'b'])
    .withColumn('ratio', try_divide(col('a'), col('b')))
    # .withColumn('ratio', col('a') / col('b'))
).show()

### Task 7: Invalid cast

Casting a non-numeric string to `int` fails under ANSI. `try_cast` returns `null` instead.

Hint:
* failing: `CAST('not a number' AS INT)`
* safe: `try_cast(col, IntegerType())` or `TRY_CAST(... AS INT)` in SQL

In [ ]:
# Throws an error with ansi enabled

# spark.sql("SELECT CAST('not a number' AS INT) AS x").show()

In [ ]:
(
    spark
    .createDataFrame([('123',), ('  42 ',), ('not a number',), (None,)], ['raw'])
    .withColumn('parsed', (col('raw').try_cast('int')))
).show()

### Task 8: Arithmetic overflow

Multiply `reputation` (an `int`) by a billion. For almost any non-trivial reputation value, the result overflows 32-bit signed int and ANSI mode raises an `ArithmeticException`. Then use `try_multiply` to get `null` back instead — the rest of the job survives.

Hint:
* the failing expression: `col('reputation').cast('int') * lit(1_000_000_000).cast('int')`
* the safe version: `try_multiply(col('reputation').cast('int'), lit(1_000_000_000).cast('int'))`
* siblings: `try_add`, `try_subtract`, `try_mod`

In [ ]:
# Throws an error with ansi enabled:

(
    usersDF
    .withColumn('inflated', col('reputation').cast('int') * lit(1_000_000_000).cast('int'))
    .select('user_id', 'reputation', 'inflated')
    .orderBy(col('reputation').desc())
).show(5)

In [ ]:
(
    usersDF
    .withColumn('inflated', try_multiply(col('reputation').cast('int'), lit(1_000_000_000).cast('int')))
    .select('user_id', 'reputation', 'inflated')
    .orderBy(col('reputation').desc())
).show(5)

### Task 9: Lenient numeric parsing with `try_to_number`

`try_to_number(value, format)` parses a formatted numeric string (locale-style thousands separators, currency, sign, etc.) and returns `null` for inputs that do not match the format — useful when ingesting messy upstream data without throwing the whole micro-batch away.

Hint:
* `to_number`/`try_to_number` use a format string with patterns like `9`, `0`, `,`, `.`, `$`, `S`, ...
* docs for [number format strings](https://spark.apache.org/docs/latest/sql-ref-number-pattern.html)

In [ ]:
(
    spark
    .createDataFrame([('1,234.56',), ('99.00',), ('garbage',), (None,)], ['raw'])
    .withColumn('parsed', try_to_number(col('raw'), lit('9,999.99')))
).show()

In [ ]:
spark.stop()